# 열 줄로 읽는 트랜스포머 — 실행용 노트북

설치 없이 위에서부터 셀을 실행하면 학습이 돌아갑니다.
런타임은 **CPU 로 충분합니다.**

이 파일의 코드가 책이 처음부터 끝까지 읽어 내는 그 코드입니다. 백 줄이 조금 넘고,
학습·검증·생성이 다 들어 있으며, 모듈 수준 변수가 없습니다.
온라인에 공개된 nanoGPT 를 바탕으로 삼아 파이토치로 대대적으로 리팩토링한 파일입니다.

저장소 · https://github.com/YoungjuneKwon/ten-line-transformer

권영준

## 1. 데이터 — 셰익스피어 텍스트 한 파일 (약 100만 자)

In [ ]:
!wget -q -O input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
print(open('input.txt').read()[:200])

## 2. 코드 전문

책이 다루는 파일입니다. 아래 열 줄이 학습 한 스텝이고, 책의 각 장이 그중 한 줄씩을 엽니다.

```python
# ── 한 번만 한다
train_data, val_data, c2i, i2c, V = load_data("input.txt")   # 텍스트 → 번호 배열
model = Transformer(V, T, C, H, n_block)                 # wte·wpe·blocks·wue

# ── 여기서부터 매 스텝 반복한다
tokens, answers = get_batch(train_data, B, T)   # (B, T), (B, T)      ①
optimizer.zero_grad(set_to_none=True)
x = wte[tokens] + wpe[:T]                       # (B, T, C)           ②
for blk in blocks:                              # n_block 번          ③
    x = x + blk.attn(layer_norm(x))             # (B, T, C)
    x = x + blk.mlp(layer_norm(x))              # (B, T, C)
logits = layer_norm(x) @ wue                    # (B, T, V)           ④
loss = cross_entropy(logits, answers)           # 스칼라 1개
loss.backward()
optimizer.step()
```

위 열 줄은 한 스텝을 펼쳐 놓은 것입니다. 실제 파일에서는 두 자리에 나뉘어 있어요.
가운데 여섯 줄은 `Transformer.forward()` 안에 있고, 나머지 넷은 `train()` 의 학습 루프에
있습니다. `forward` 는 정답(`answers`)을 함께 받으면 손실까지 계산해 돌려줍니다 —
허깅페이스 `GPT2LMHeadModel.forward(..., labels=None)` 이나 nanoGPT 의
`forward(idx, targets=None)` 과 같은 방식입니다. 정답을 주지 않으면 손실 자리에
`None` 이 옵니다.

```python
  def forward(self, tokens, answers=None):
    x = self.wte[tokens] + self.wpe[:tokens.shape[1]]
    for blk in self.blocks:
      x = x + blk.attn(layer_norm(x))
      x = x + blk.mlp(layer_norm(x))
    logits = layer_norm(x) @ self.wue
    return logits, None if answers is None else cross_entropy(logits, answers)
```

```python
  for step in range(steps + 1):
    tokens, answers = get_batch(train_data, B, T)
    optimizer.zero_grad(set_to_none=True)
    _, loss = model(tokens, answers)
    loss.backward()
    optimizer.step()
```

아래 「코드 전문」 셀의 그 줄들입니다. 옮겨 적은 것이 아니라 같은 파일에서 뽑았습니다.

In [ ]:
import sys, math, time, torch, torch.nn as nn
from torch.nn import functional as F

def load_data(path, train_ratio=0.9):
  text = open(path, encoding="utf-8").read()
  chars = sorted(set(text))
  c2i = {c: i for i, c in enumerate(chars)}
  data = torch.tensor([c2i[c] for c in text], dtype=torch.long)
  cut = int(train_ratio * len(data))
  return data[:cut], data[cut:], c2i, {i: c for c, i in c2i.items()}, len(chars)

def get_batch(data, B, T):
  starts = torch.randint(len(data) - T - 1, (B,))
  tokens = torch.stack([data[s:s + T] for s in starts])
  answers = torch.stack([data[s + 1:s + T + 1] for s in starts])
  return tokens, answers

def layer_norm(x, eps=1e-5):
  return (x - x.mean(-1, keepdim=True)) / torch.sqrt(x.var(-1, unbiased=False, keepdim=True) + eps)

def cross_entropy(logits, answers):
  return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), answers.reshape(-1))

class Attention(nn.Module):
  def __init__(self, T, C, H):
    super().__init__()
    self.H = H
    self.hs = C // H
    self.qkv = nn.Linear(C, 3 * H * self.hs, bias=False)
    self.proj = nn.Linear(C, C, bias=False)
    self.register_buffer("mask", torch.tril(torch.ones(T, T)).view(1, 1, T, T))

  def forward(self, x):
    b, t, c = x.shape
    q, k, v = [z.view(b, t, self.H, self.hs).transpose(1, 2)
               for z in self.qkv(x).split(c, dim=2)]
    att = (q @ k.transpose(-2, -1)) / math.sqrt(self.hs)
    att = att.masked_fill(self.mask[:, :, :t, :t] == 0, float("-inf"))
    y = F.softmax(att, dim=-1) @ v
    y = y.transpose(1, 2).contiguous().view(b, t, c)
    return self.proj(y)

class MLP(nn.Module):
  def __init__(self, C):
    super().__init__()
    self.fc = nn.Linear(C, 4 * C, bias=False)
    self.out = nn.Linear(4 * C, C, bias=False)

  def forward(self, x):
    return self.out(F.gelu(self.fc(x)))

class Block(nn.Module):
  def __init__(self, T, C, H):
    super().__init__()
    self.attn = Attention(T, C, H)
    self.mlp = MLP(C)

class Transformer(nn.Module):
  def __init__(self, V, T, C, H, n_block):
    super().__init__()
    self.wte = nn.Parameter(torch.randn(V, C) * 0.02)
    self.wpe = nn.Parameter(torch.randn(T, C) * 0.02)
    self.wue = nn.Parameter(torch.randn(C, V) * 0.02)
    self.blocks = nn.ModuleList([Block(T, C, H) for _ in range(n_block)])

  def forward(self, tokens, answers=None):
    x = self.wte[tokens] + self.wpe[:tokens.shape[1]]
    for blk in self.blocks:
      x = x + blk.attn(layer_norm(x))
      x = x + blk.mlp(layer_norm(x))
    logits = layer_norm(x) @ self.wue
    return logits, None if answers is None else cross_entropy(logits, answers)

def report(model, step, loss, t0, val_batch):
  val_tokens, val_answers = val_batch
  with torch.no_grad():
    _, val_loss = model(val_tokens, val_answers)
  print(f"{step:6d}{loss.item():10.4f}{val_loss.item():10.4f}{time.time() - t0:8.1f}")

def train(model, train_data, val_data, steps, B, T):
  optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4,
                                betas=(0.9, 0.95), weight_decay=0.1)
  print(f"{'step':>6}{'train':>10}{'val':>10}{'sec':>8}")
  report_every = max(1, steps // 10)
  t0 = time.time()
  for step in range(steps + 1):
    tokens, answers = get_batch(train_data, B, T)
    optimizer.zero_grad(set_to_none=True)
    _, loss = model(tokens, answers)
    loss.backward()
    optimizer.step()
    if step % report_every == 0:
      val_batch = get_batch(val_data, B, T)
      report(model, step, loss, t0, val_batch)

def generate(model, n_chars, T, c2i, i2c, seed_text="\n", temperature=0.8):
  seq = torch.tensor([[c2i[c] for c in seed_text]], dtype=torch.long)
  with torch.no_grad():
    for _ in range(n_chars):
      logits, _ = model(seq[:, -T:])
      probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
      seq = torch.cat([seq, torch.multinomial(probs, 1)], dim=1)
  return "".join(i2c[i] for i in seq[0].tolist())

def cli():
  steps = int(sys.argv[1]) if len(sys.argv) > 1 else 2000
  corpus = sys.argv[2] if len(sys.argv) > 2 else "input.txt"
  return steps, corpus

def main(steps, corpus):
  B, T, C, H, n_block = 32, 64, 128, 4, 4
  assert C % H == 0, "C must be divisible by H"
  torch.manual_seed(1337)

  train_data, val_data, c2i, i2c, V = load_data(corpus)
  model = Transformer(V, T, C, H, n_block)
  train(model, train_data, val_data, steps, B, T)
  print("\nsample")
  print(generate(model, 800, T, c2i, i2c))
  return model

## 3. 돌려 보기 — 200 스텝

`0: 4.2006 / 3.9836` 에서 시작해 `200: 2.5400 / 2.4785` 로 내려갑니다.
시드를 1337 로 고정했으므로 README 에 적힌 숫자가 그대로 나옵니다.

`sec` 는 기기마다 다릅니다. 손실 값은 다르지 않습니다.

In [ ]:
model = main(200, 'input.txt')

## 4. 2000 스텝

셰익스피어 흉내가 나오기 시작합니다. 글자 단위로 다음 글자만 맞히도록 훈련한 모델이
이름 뒤에 콜론을 찍고 줄을 바꿉니다.

하나 예상하고 보십시오 — 여기서는 200 스텝 자리가 `2.5400` 이 아니라 `2.4812` 로
읽힙니다. 보고가 20 마다가 아니라 200 마다 떨어지고, 보고할 때마다 학습 묶음과 같은
난수원에서 검증 묶음을 뽑기 때문입니다. 같은 시드, 같은 코드, 다른 뽑기 일정입니다.
대조는 같은 명령끼리 합니다.

In [ ]:
model = main(2000, 'input.txt')

## 5. 바꿔 보기

`main` 첫 줄의 다섯 숫자가 이 모델의 크기 전부입니다.
`C` 를 64 로 줄이면 파라미터가 4분의 1이 되고, 256 으로 올리면 손실이 더 내려가는 대신
시간이 다섯 배 듭니다. 책 4장이 그 표를 싣습니다.

In [ ]:
# B, T, C, H, n_block = 32, 64, 128, 4, 4   <- main 안의 기본값
# 아래를 고쳐 직접 확인해 보십시오.
import torch, time
B, T, C, H, n_block = 32, 64, 64, 4, 4
torch.manual_seed(1337)
train_data, val_data, c2i, i2c, V = load_data('input.txt')
m = Transformer(V, T, C, H, n_block)
print('파라미터', sum(p.numel() for p in m.parameters()))
train(m, train_data, val_data, 200, B, T)